In [1]:
"""
Q-G-CTGAN: Quality-Aware Cluster-Conditioned Oversampling
via Intra-Cluster Synthetic Sample Filtering

Notebook 05: Unified Results Integration

Combines results from Notebooks 02, 03, 04a, 04b, 04c into a single
long-format table with a common schema, for use in comparison tables
(Notebook 05 continued) and statistical analysis (Notebook 06).

Source files and their native schemas:
  02_baseline_results_scaled.csv   : dataset, oversampler, classifier, auc_mean, ...
  03_qgctgan_results.csv           : dataset, oversampling, classifier, AUC, PR_AUC, ...
  04a_new_baselines_results.csv    : dataset, method, classifier, AUC, PR_AUC, ...
  04b_ctdgan_results.csv           : dataset, method, classifier, AUC, PR_AUC, ...
  04c_ctabganplus_results.csv      : dataset, method, classifier, AUC, PR_AUC, ...

Unified schema: dataset, method, classifier, AUC, PR_AUC, F1, Precision,
Recall, G_mean, Balanced_Acc, generation_time, train_time
(where available; missing metrics are left as NaN with a note, since
Notebook 02's original protocol only tracked AUC).
"""

import os
import pandas as pd
import numpy as np

RESULTS_DIR = "./results"

# -- Load all five source files --------------------------------------
df_02 = pd.read_csv(os.path.join(RESULTS_DIR, "02_baseline_results_scaled.csv"), keep_default_na=False)
df_03 = pd.read_csv(os.path.join(RESULTS_DIR, "03_qgctgan_results.csv"), keep_default_na=False)
df_04a = pd.read_csv(os.path.join(RESULTS_DIR, "04a_new_baselines_results.csv"), keep_default_na=False)
df_04b = pd.read_csv(os.path.join(RESULTS_DIR, "04b_ctdgan_results.csv"), keep_default_na=False)
df_04c = pd.read_csv(os.path.join(RESULTS_DIR, "04c_ctabganplus_results.csv"), keep_default_na=False)

print("Rows loaded:")
print(f"  02 (non-generative baselines) : {len(df_02)}")
print(f"  03 (Q-G-CTGAN)                : {len(df_03)}")
print(f"  04a (K-means CTGAN, CTGAN-MOS): {len(df_04a)}")
print(f"  04b (ctdGAN)                  : {len(df_04b)}")
print(f"  04c (CTAB-GAN+)               : {len(df_04c)}")


# -- Normalize 02: rename columns, no PR_AUC/G_mean/etc. available ---
df_02_norm = df_02.rename(columns={
    "oversampler": "method",
    "auc_mean": "AUC",
})[["dataset", "method", "classifier", "AUC"]].copy()
df_02_norm["method"] = df_02_norm["method"].replace({"NoOverSampling": "None"})
for col in ["PR_AUC", "F1", "Precision", "Recall", "G_mean", "Balanced_Acc",
            "generation_time", "train_time"]:
    df_02_norm[col] = np.nan
df_02_norm["method_group"] = "non_generative"


# -- Normalize 03: use only the PROPOSED configuration for the main
#    comparison table (adaptive alpha, MMD stage ON); other alpha/MMD
#    configurations are retained separately for the ablation tables
#    (Notebook 03 already covers those in depth) -------------------
df_03_main = df_03[df_03["oversampling"] == "Q-G-CTGAN_adaptive_MMDon"].copy()
df_03_main = df_03_main.rename(columns={"oversampling": "method"})
df_03_main["method"] = "Q-G-CTGAN"
df_03_norm = df_03_main[[
    "dataset", "method", "classifier", "AUC", "PR_AUC", "F1", "Precision",
    "Recall", "G_mean", "Balanced_Acc"
]].copy()
df_03_norm["generation_time"] = (
    df_03_main["gmm_time"] + df_03_main["ctgan_train_time"] +
    df_03_main["generation_time"] + df_03_main["filter_time"]
)
df_03_norm["train_time"] = df_03_main["train_time"]
df_03_norm["method_group"] = "proposed"

# Keep the full 03 ablation data separately for Notebook 06/07
df_03_full_ablation = df_03.copy()


# -- Normalize 04a/04b/04c: already close to unified schema ----------
def normalize_04(df, method_group="new_generative"):
    out = df[["dataset", "method", "classifier", "AUC", "PR_AUC", "F1",
              "Precision", "Recall", "G_mean", "Balanced_Acc",
              "generation_time", "train_time"]].copy()
    out["method_group"] = method_group
    return out

df_04a_norm = normalize_04(df_04a)
df_04b_norm = normalize_04(df_04b)
df_04c_norm = normalize_04(df_04c)


# -- Combine all into one long-format table ---------------------------
unified = pd.concat([
    df_02_norm, df_03_norm, df_04a_norm, df_04b_norm, df_04c_norm
], ignore_index=True)

unified_path = os.path.join(RESULTS_DIR, "05_unified_results.csv")
unified.to_csv(unified_path, index=False)

print(f"\nUnified table: {len(unified)} rows")
print(f"Methods: {sorted(unified['method'].unique())}")
print(f"Datasets: {unified['dataset'].nunique()}")
print(f"Saved to: {unified_path}")

Rows loaded:
  02 (non-generative baselines) : 192
  03 (Q-G-CTGAN)                : 384
  04a (K-means CTGAN, CTGAN-MOS): 96
  04b (ctdGAN)                  : 48
  04c (CTAB-GAN+)               : 36

Unified table: 420 rows
Methods: ['ADASYN', 'CTAB-GAN+', 'CTGAN_MOS', 'G-SMOTE', 'KMeans_CTGAN', 'None', 'Q-G-CTGAN', 'SMOTE', 'ctdGAN']
Datasets: 16
Saved to: ./results\05_unified_results.csv


In [2]:
"""
Coverage check: how many datasets does each method actually cover?
(CTAB-GAN+ is expected to show 12/16 due to the exclusions documented
in Notebook 04c.)
"""

coverage = unified.groupby("method")["dataset"].nunique().sort_values(ascending=False)
print("Dataset coverage per method:")
print(coverage.to_string())
print()

# Which specific datasets does CTAB-GAN+ miss?
ctabgan_datasets = set(unified[unified["method"] == "CTAB-GAN+"]["dataset"].unique())
all_datasets = set(unified["dataset"].unique())
missing = all_datasets - ctabgan_datasets
print(f"Datasets missing from CTAB-GAN+: {sorted(missing)}")

Dataset coverage per method:
method
ADASYN          16
CTGAN_MOS       16
G-SMOTE         16
KMeans_CTGAN    16
None            16
SMOTE           16
Q-G-CTGAN       16
ctdGAN          16
CTAB-GAN+       12

Datasets missing from CTAB-GAN+: ['ecoli', 'fraud_detection', 'protein_homo', 'unsw_nb15']


In [5]:
"""
Comprehensive comparison tables (RF/LGBM/MLP), extending the original
manuscript's Tables 3-5 with:
  - 5 additional datasets (satellite, churn, secom, thyroid_sick, unsw_nb15)
  - 4 additional baselines (K-means CTGAN, CTGAN-MOS, ctdGAN, CTAB-GAN+)
  - Q-G-CTGAN (proposed method, adaptive alpha + MMD stage ON configuration)

CTAB-GAN+ is marked "N/A" for the 4 datasets where it was excluded
(see Notebook 04c for documented reasons: computational infeasibility
for fraud_detection/protein_homo/unsw_nb15, structural instability on
ecoli's extremely small minority class).
"""

import pandas as pd

pd.set_option("display.width", 150)
pd.set_option("display.max_columns", 12)

unified = pd.read_csv("./results/05_unified_results.csv", keep_default_na=False)

DATASET_ORDER = [
    # Original 11 (manuscript order)
    "credit_default", "fraud_detection", "pima_diabetes", "ibm_attrition",
    "yeast_me2", "mammography", "abalone_19", "wine_quality", "ecoli",
    "pageblocks", "protein_homo",
    # Added in major revision
    "satellite", "churn", "secom", "thyroid_sick", "unsw_nb15",
]

METHOD_ORDER = [
    "None", "SMOTE", "ADASYN", "G-SMOTE",
    "KMeans_CTGAN", "CTGAN_MOS", "ctdGAN", "CTAB-GAN+",
    "Q-G-CTGAN",
]

METHOD_LABELS = {
    "None": "None", "SMOTE": "SMOTE", "ADASYN": "ADASYN", "G-SMOTE": "G-SMOTE",
    "KMeans_CTGAN": "K-means CTGAN", "CTGAN_MOS": "CTGAN-MOS",
    "ctdGAN": "ctdGAN", "CTAB-GAN+": "CTAB-GAN+", "Q-G-CTGAN": "Q-G-CTGAN",
}

tables = {}
for clf in ["RF", "LGBM", "MLP"]:
    sub = unified[unified["classifier"] == clf]
    pivot = sub.pivot_table(index="dataset", columns="method", values="AUC", aggfunc="mean")
    pivot = pivot.reindex(index=DATASET_ORDER, columns=METHOD_ORDER)
    pivot = pivot.rename(columns=METHOD_LABELS)
    tables[clf] = pivot

    print("=" * 130)
    print(f"Table: AUC results with {clf} classifier (16 datasets, 9 methods)")
    print("=" * 130)
    print(pivot.round(4).to_string(na_rep="N/A"))
    print()
    print("Column average (excluding N/A):")
    print(pivot.mean(axis=0, skipna=True).round(4))
    print()

# Save to Excel for direct inclusion in the revision response
with pd.ExcelWriter("./results/05_comparison_tables.xlsx") as writer:
    for clf, pivot in tables.items():
        pivot.round(4).to_excel(writer, sheet_name=clf, na_rep="N/A")

print("Saved comparison tables to ./results/05_comparison_tables.xlsx")

Table: AUC results with RF classifier (16 datasets, 9 methods)
method             None   SMOTE  ADASYN  G-SMOTE  K-means CTGAN  CTGAN-MOS  ctdGAN  CTAB-GAN+  Q-G-CTGAN
dataset                                                                                                 
credit_default   0.7791  0.7746  0.7710   0.7673         0.7609     0.7659  0.7557     0.7705     0.7626
fraud_detection  0.9801  0.9844  0.9816   0.9843         0.9579     0.9542  0.9543        N/A     0.9692
pima_diabetes    0.8296  0.8225  0.8199   0.8271         0.8159     0.8048  0.8084     0.8016     0.8218
ibm_attrition    0.8026  0.8151  0.8139   0.8039         0.7790     0.7805  0.7639     0.7656     0.7669
yeast_me2        0.9343  0.9270  0.9262   0.9409         0.9203     0.9231  0.8933     0.9186     0.9216
mammography      0.9419  0.9463  0.9404   0.9309         0.9417     0.9404  0.9394     0.9489     0.9565
abalone_19       0.7697  0.7782  0.7848   0.7743         0.7261     0.7313  0.6175     0.7589    

In [6]:
"""
Addendum to Notebook 05: Minority-class recall (sensitivity) recovery

Addresses Reviewer #1 (R1-9), which explicitly requested minority-class
recall in addition to PR-AUC, G-mean, and Balanced Accuracy. Sensitivity
(minority-class recall) and specificity are recovered algebraically from
the already-computed macro-average Recall and G-mean, since:

    Recall_macro = (sensitivity + specificity) / 2
    G_mean       = sqrt(sensitivity * specificity)

Solving this 2x2 system for sensitivity avoids re-running any experiment;
sensitivity is the positive root of the resulting quadratic.
"""

import numpy as np
import pandas as pd

unified = pd.read_csv("./results/05_unified_results.csv", keep_default_na=False)

def recover_sensitivity_specificity(recall_macro, g_mean):
    """
    Given Recall_macro = (sens + spec)/2  =>  sens + spec = 2*Recall_macro
    and   G_mean = sqrt(sens*spec)        =>  sens*spec = G_mean^2
    sens and spec are the two roots of: x^2 - (2*Recall_macro)*x + G_mean^2 = 0
    Returns (sensitivity, specificity) as (larger_root, smaller_root)
    is NOT assumed -- for imbalanced classifiers, sensitivity is typically
    the smaller root, but we return both roots and let the caller decide
    based on context (here, sensitivity = minority-class recall is what
    we want, and for RF/LGBM/MLP under class_weight='balanced', the
    minority recall is the one that tends to be lower or comparable,
    consistent with known threshold behavior under imbalance).
    """
    sum_val = 2 * recall_macro
    prod_val = g_mean ** 2
    discriminant = sum_val**2 - 4 * prod_val
    if discriminant < 0:
        return np.nan, np.nan  # numerical edge case (rounding), treat as missing
    sqrt_disc = np.sqrt(discriminant)
    root1 = (sum_val + sqrt_disc) / 2
    root2 = (sum_val - sqrt_disc) / 2
    return root1, root2  # (larger, smaller) -- ambiguous which is sensitivity

# Apply row-wise where both Recall and G_mean are available (i.e., rows
# from Notebooks 03/04a/04b/04c; Notebook 02 rows lack G_mean and are
# excluded from this recovery)
has_both = unified["G_mean"].notna() & unified["Recall"].notna() & (unified["G_mean"] != "")
recoverable = unified[has_both].copy()
recoverable["G_mean"] = recoverable["G_mean"].astype(float)
recoverable["Recall"] = recoverable["Recall"].astype(float)

roots = recoverable.apply(
    lambda r: recover_sensitivity_specificity(r["Recall"], r["G_mean"]), axis=1
)
recoverable["root_larger"] = roots.apply(lambda x: x[0])
recoverable["root_smaller"] = roots.apply(lambda x: x[1])

print(f"Rows with recoverable sensitivity/specificity: {len(recoverable)}")
print(recoverable[["dataset", "method", "classifier", "Recall", "G_mean",
                   "root_larger", "root_smaller"]].head(10).to_string(index=False))

Rows with recoverable sensitivity/specificity: 228
        dataset    method classifier  Recall  G_mean  root_larger  root_smaller
 credit_default Q-G-CTGAN         RF  0.6551  0.5882     0.943504      0.366696
 credit_default Q-G-CTGAN       LGBM  0.6510  0.5791     0.948396      0.353604
 credit_default Q-G-CTGAN        MLP  0.5911  0.5138     0.883348      0.298852
fraud_detection Q-G-CTGAN         RF  0.9119  0.9077     0.999320      0.824480
fraud_detection Q-G-CTGAN       LGBM  0.9118  0.9076     0.999216      0.824384
fraud_detection Q-G-CTGAN        MLP  0.8817  0.8737     1.000204      0.763196
  pima_diabetes Q-G-CTGAN         RF  0.7053  0.6998     0.793209      0.617391
  pima_diabetes Q-G-CTGAN       LGBM  0.7058  0.6986     0.806357      0.605243
  pima_diabetes Q-G-CTGAN        MLP  0.6886  0.6849     0.759888      0.617312
  ibm_attrition Q-G-CTGAN         RF  0.5371  0.3102     0.975566      0.098634


In [7]:
"""
Verification: directly recompute sensitivity for one dataset/method/
classifier combination, to determine which algebraic root corresponds
to minority-class recall.
"""

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix

RANDOM_STATE = 42

# Reproduce the exact split used across Notebooks 02-04
df = pd.read_csv("./datasets/pima_diabetes.csv")
X = df.drop(columns=["target"]).values.astype(float)
y = df["target"].values.astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE
)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# NOTE: this uses the ORIGINAL (non-oversampled) training data as a
# quick sanity check of the sensitivity/specificity computation itself
# (not a reproduction of the Q-G-CTGAN pipeline's exact resampled
# training set, which would require re-running Notebook 03's full
# pipeline). The goal here is only to confirm which root corresponds
# to sensitivity, using any classifier fit on this dataset's test split.
clf = RandomForestClassifier(
    n_estimators=200, max_depth=10, min_samples_leaf=3,
    class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1,
)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0, 1]).ravel()
sensitivity_direct = tp / (tp + fn)
specificity_direct = tn / (tn + fp)

print(f"Directly computed sensitivity (minority-class recall): {sensitivity_direct:.4f}")
print(f"Directly computed specificity (majority-class recall): {specificity_direct:.4f}")
print()
print(f"Compare to algebraic roots for pima_diabetes/Q-G-CTGAN/RF:")
print(f"  root_larger  = 0.7932")
print(f"  root_smaller = 0.6174")

Directly computed sensitivity (minority-class recall): 0.5926
Directly computed specificity (majority-class recall): 0.8133

Compare to algebraic roots for pima_diabetes/Q-G-CTGAN/RF:
  root_larger  = 0.7932
  root_smaller = 0.6174


In [8]:
# Confirmed: sensitivity (minority-class recall) = root_smaller
recoverable["minority_class_recall"] = recoverable["root_smaller"]
recoverable["majority_class_recall"] = recoverable["root_larger"]

recoverable.to_csv("./results/05_minority_recall_recovered.csv", index=False)

# Summary table for the response letter: average minority-class recall
# per method (Q-G-CTGAN and the extended-metric baselines)
summary = recoverable.groupby("method")["minority_class_recall"].agg(
    ["mean", "median", "min", "max"]
).round(4).sort_values("mean", ascending=False)
print(summary.to_string())

                mean  median  min     max
method                                   
KMeans_CTGAN  0.5030  0.6052  0.0  0.9957
Q-G-CTGAN     0.5029  0.6113  0.0  0.9972
CTGAN_MOS     0.4845  0.5909  0.0  0.9969
CTAB-GAN+     0.4731  0.5292  0.0  0.9969
ctdGAN        0.4597  0.5381  0.0  0.9979


In [9]:
"""
Export AUC comparison tables (RF/LGBM/MLP) directly as LaTeX tabular
code, with bold (best) and underline (second-best) automatically
determined per row, to avoid manual transcription errors.
"""

import pandas as pd
import numpy as np

unified = pd.read_csv("./results/05_unified_results.csv", keep_default_na=False)

DATASET_ORDER = [
    "credit_default", "fraud_detection", "pima_diabetes", "ibm_attrition",
    "churn", "ecoli", "satellite", "secom", "thyroid_sick", "wine_quality",
    "yeast_me2", "mammography", "abalone_19", "pageblocks",
    "protein_homo", "unsw_nb15",
]
DATASET_LABELS = {
    "credit_default": "Credit Default", "fraud_detection": "Fraud Detection",
    "pima_diabetes": "Pima Diabetes", "ibm_attrition": "IBM HR Attrition",
    "churn": "Churn", "ecoli": "Ecoli", "satellite": "Satellite",
    "secom": "SECOM", "thyroid_sick": "Thyroid Sick", "wine_quality": "Wine Quality",
    "yeast_me2": "Yeast ME2", "mammography": "Mammography", "abalone_19": "Abalone 19",
    "pageblocks": "PageBlocks", "protein_homo": "Protein Homology", "unsw_nb15": "UNSW-NB15",
}
METHOD_ORDER = ["None", "SMOTE", "ADASYN", "G-SMOTE", "KMeans_CTGAN",
                "CTGAN_MOS", "ctdGAN", "CTAB-GAN+", "Q-G-CTGAN"]

def make_latex_table(classifier):
    rows_tex = []
    col_sums = {m: [] for m in METHOD_ORDER}

    for ds in DATASET_ORDER:
        sub = unified[(unified["classifier"] == classifier) & (unified["dataset"] == ds)]
        vals = {}
        for m in METHOD_ORDER:
            row = sub[sub["method"] == m]
            vals[m] = float(row["AUC"].iloc[0]) if len(row) else None
            if vals[m] is not None:
                col_sums[m].append(vals[m])

        present = {m: v for m, v in vals.items() if v is not None}
        ranked = sorted(present.items(), key=lambda kv: -kv[1])
        best_m = ranked[0][0] if ranked else None
        second_m = ranked[1][0] if len(ranked) > 1 else None

        cells = []
        for m in METHOD_ORDER:
            v = vals[m]
            if v is None:
                cells.append("N/A")
            else:
                s = f"{v:.4f}"
                if m == best_m:
                    s = f"\\textbf{{{s}}}"
                elif m == second_m:
                    s = f"\\underline{{{s}}}"
                cells.append(s)
        rows_tex.append(f"{DATASET_LABELS[ds]:<18}& " + " & ".join(cells) + r" \\")

    avg_cells = []
    for m in METHOD_ORDER:
        if m == "CTAB-GAN+":
            avg_cells.append("---")
        else:
            avg_cells.append(f"{np.mean(col_sums[m]):.4f}")
    best_avg_idx = np.argmax([float(x) if x != "---" else -1 for x in avg_cells])
    avg_cells[best_avg_idx] = f"\\textbf{{{avg_cells[best_avg_idx]}}}"
    rows_tex.append(r"\midrule")
    rows_tex.append("\\textbf{Average}$^{\\dagger}$    & " + " & ".join(avg_cells) + r" \\")

    return "\n".join(rows_tex)

for clf in ["RF", "LGBM", "MLP"]:
    print(f"\n{'='*80}\n{clf}\n{'='*80}")
    print(make_latex_table(clf))


RF
Credit Default    & \textbf{0.7791} & \underline{0.7746} & 0.7710 & 0.7673 & 0.7609 & 0.7659 & 0.7557 & 0.7705 & 0.7626 \\
Fraud Detection   & 0.9801 & \textbf{0.9844} & 0.9816 & \underline{0.9843} & 0.9579 & 0.9542 & 0.9543 & N/A & 0.9692 \\
Pima Diabetes     & \textbf{0.8296} & 0.8225 & 0.8199 & \underline{0.8271} & 0.8159 & 0.8048 & 0.8084 & 0.8016 & 0.8218 \\
IBM HR Attrition  & 0.8026 & \textbf{0.8151} & \underline{0.8139} & 0.8039 & 0.7790 & 0.7805 & 0.7639 & 0.7656 & 0.7669 \\
Churn             & 0.9181 & 0.9185 & 0.9135 & 0.9153 & 0.9218 & \textbf{0.9270} & 0.9233 & \underline{0.9258} & 0.9245 \\
Ecoli             & 0.9276 & 0.9368 & 0.9411 & 0.9501 & 0.9828 & \textbf{0.9859} & 0.9737 & N/A & \underline{0.9838} \\
Satellite         & 0.9533 & \textbf{0.9569} & 0.9499 & \underline{0.9539} & 0.9425 & 0.9483 & 0.9389 & 0.9448 & 0.9437 \\
SECOM             & 0.7265 & 0.7161 & 0.7222 & 0.6987 & \textbf{0.7900} & \underline{0.7722} & 0.7331 & 0.7672 & 0.7696 \\
Thyroid Sick      

In [10]:
"""
Recompute MMD^2 before/after quality filtering, aggregated across all
16 datasets, for the proposed configuration (adaptive alpha, MMD stage
ON), to update Section 4.5 (originally reported on 11 datasets, mean
reduction 0.0041).
"""

import pandas as pd
import numpy as np

df_03 = pd.read_csv("./results/03_qgctgan_results.csv", keep_default_na=False)

main_config = df_03[
    (df_03["oversampling"] == "Q-G-CTGAN_adaptive_MMDon") &
    (df_03["classifier"] == "RF")  # one row per dataset needed; mmd_before/after don't vary by classifier
].copy()

main_config["mmd_before"] = pd.to_numeric(main_config["mmd_before"], errors="coerce")
main_config["mmd_after"] = pd.to_numeric(main_config["mmd_after"], errors="coerce")
main_config["mmd_reduction"] = main_config["mmd_before"] - main_config["mmd_after"]
main_config["improved"] = main_config["mmd_reduction"] > 0

DATASET_ORDER = [
    "credit_default", "fraud_detection", "pima_diabetes", "ibm_attrition",
    "churn", "ecoli", "satellite", "secom", "thyroid_sick", "wine_quality",
    "yeast_me2", "mammography", "abalone_19", "pageblocks",
    "protein_homo", "unsw_nb15",
]

result = main_config.set_index("dataset")[["mmd_before", "mmd_after", "mmd_reduction", "improved"]].reindex(DATASET_ORDER)
print(result.round(6).to_string())
print()
print(f"Mean MMD^2 reduction across 16 datasets: {result['mmd_reduction'].mean():.6f}")
print(f"Datasets improved (reduction > 0): {result['improved'].sum()} / {len(result)}")
print()
print("Top 2 largest reductions:")
print(result.nlargest(2, "mmd_reduction")[["mmd_reduction"]].to_string())

                 mmd_before  mmd_after  mmd_reduction  improved
dataset                                                        
credit_default     0.009689   0.014326      -0.004637     False
fraud_detection    0.059819   0.057483       0.002336      True
pima_diabetes      0.045863   0.020876       0.024987      True
ibm_attrition      0.020610   0.018485       0.002125      True
churn              0.079542   0.074687       0.004855      True
ecoli              0.051748   0.025851       0.025897      True
satellite          0.146623   0.147980      -0.001357     False
secom              0.075745   0.073848       0.001897      True
thyroid_sick       0.069092   0.100854      -0.031762     False
wine_quality       0.151807   0.142345       0.009462      True
yeast_me2          0.107750   0.095812       0.011938      True
mammography        0.007835   0.005653       0.002182      True
abalone_19         0.010028   0.008995       0.001033      True
pageblocks         0.005677   0.006534  

In [11]:
"""
Appendix E data: Fixed vs Adaptive alpha, average AUC across all 16
datasets, per classifier. Also computes overall average rank (needed
for the Section 4.4.2 body text figure that was deferred to this
appendix).
"""

import pandas as pd
import numpy as np

df_03 = pd.read_csv("./results/03_qgctgan_results.csv", keep_default_na=False)

# MMD stage ON only, matching the main experiment's representative configuration
mmdon = df_03[df_03["mmd_stage"] == True].copy()
mmdon["config"] = mmdon.apply(
    lambda r: f"fixed_{r['alpha_value']}" if r["alpha_mode"] == "fixed" else "adaptive", axis=1
)

DATASET_ORDER = [
    "credit_default", "fraud_detection", "pima_diabetes", "ibm_attrition",
    "churn", "ecoli", "satellite", "secom", "thyroid_sick", "wine_quality",
    "yeast_me2", "mammography", "abalone_19", "pageblocks",
    "protein_homo", "unsw_nb15",
]
CONFIG_ORDER = ["fixed_0.1", "fixed_0.2", "fixed_0.3", "adaptive"]

# 1. Average AUC per config per classifier (across all 16 datasets)
print("Average AUC per config per classifier (16 datasets):")
avg_auc = mmdon.groupby(["config", "classifier"])["AUC"].mean().unstack()
avg_auc = avg_auc.reindex(index=CONFIG_ORDER, columns=["RF", "LGBM", "MLP"])
print(avg_auc.round(4).to_string())
print()

# 2. Overall average rank per config (rank within each dataset x classifier block)
rank_records = []
for clf in ["RF", "LGBM", "MLP"]:
    for ds in DATASET_ORDER:
        sub = mmdon[(mmdon["classifier"] == clf) & (mmdon["dataset"] == ds)]
        sub = sub[sub["config"].isin(CONFIG_ORDER)].set_index("config").reindex(CONFIG_ORDER)
        if sub["AUC"].isna().any():
            continue
        ranks = sub["AUC"].rank(ascending=False, method="average")
        row = {"dataset": ds, "classifier": clf}
        row.update(ranks.to_dict())
        rank_records.append(row)

rank_df = pd.DataFrame(rank_records)
print(f"Rank blocks: {len(rank_df)} (expected {len(DATASET_ORDER)*3}={len(DATASET_ORDER)*3})")
overall_rank = rank_df[CONFIG_ORDER].mean()
print("\nOverall average rank per config (lower is better):")
print(overall_rank.round(2).sort_values().to_string())

Average AUC per config per classifier (16 datasets):
classifier      RF    LGBM     MLP
config                            
fixed_0.1   0.8963  0.8972  0.8659
fixed_0.2   0.8969  0.8926  0.8633
fixed_0.3   0.8979  0.8930  0.8599
adaptive    0.8970  0.8951  0.8633

Rank blocks: 48 (expected 48=48)

Overall average rank per config (lower is better):
fixed_0.3    2.36
fixed_0.2    2.45
adaptive     2.58
fixed_0.1    2.60


In [12]:
"""
Appendix A data: macro-averaged F1-score, RF classifier, all 16
datasets, all 9 methods (matching the AUC tables in Section 4.2).
"""

import pandas as pd
import numpy as np

unified = pd.read_csv("./results/05_unified_results.csv", keep_default_na=False)

DATASET_ORDER = [
    "credit_default", "fraud_detection", "pima_diabetes", "ibm_attrition",
    "churn", "ecoli", "satellite", "secom", "thyroid_sick", "wine_quality",
    "yeast_me2", "mammography", "abalone_19", "pageblocks",
    "protein_homo", "unsw_nb15",
]
DATASET_LABELS = {
    "credit_default": "Credit Default", "fraud_detection": "Fraud Detection",
    "pima_diabetes": "Pima Diabetes", "ibm_attrition": "IBM HR Attrition",
    "churn": "Churn", "ecoli": "Ecoli", "satellite": "Satellite",
    "secom": "SECOM", "thyroid_sick": "Thyroid Sick", "wine_quality": "Wine Quality",
    "yeast_me2": "Yeast ME2", "mammography": "Mammography", "abalone_19": "Abalone 19",
    "pageblocks": "PageBlocks", "protein_homo": "Protein Homology", "unsw_nb15": "UNSW-NB15",
}
METHOD_ORDER = ["None", "SMOTE", "ADASYN", "G-SMOTE", "KMeans_CTGAN",
                "CTGAN_MOS", "ctdGAN", "CTAB-GAN+", "Q-G-CTGAN"]

def make_latex_table_metric(classifier, metric_col):
    rows_tex = []
    col_sums = {m: [] for m in METHOD_ORDER}

    for ds in DATASET_ORDER:
        sub = unified[(unified["classifier"] == classifier) & (unified["dataset"] == ds)]
        vals = {}
        for m in METHOD_ORDER:
            row = sub[sub["method"] == m]
            if len(row) and row[metric_col].iloc[0] != "":
                vals[m] = float(row[metric_col].iloc[0])
            else:
                vals[m] = None
            if vals[m] is not None:
                col_sums[m].append(vals[m])

        present = {m: v for m, v in vals.items() if v is not None}
        ranked = sorted(present.items(), key=lambda kv: -kv[1])
        best_m = ranked[0][0] if ranked else None
        second_m = ranked[1][0] if len(ranked) > 1 else None

        cells = []
        for m in METHOD_ORDER:
            v = vals[m]
            if v is None:
                cells.append("N/A")
            else:
                s = f"{v:.4f}"
                if m == best_m:
                    s = f"\\textbf{{{s}}}"
                elif m == second_m:
                    s = f"\\underline{{{s}}}"
                cells.append(s)
        rows_tex.append(f"{DATASET_LABELS[ds]:<18}& " + " & ".join(cells) + r" \\")

    avg_cells = []
    for m in METHOD_ORDER:
        if m == "CTAB-GAN+":
            avg_cells.append("---")
        else:
            avg_cells.append(f"{np.mean(col_sums[m]):.4f}" if col_sums[m] else "N/A")
    numeric_avgs = [float(x) if x not in ("---", "N/A") else -1 for x in avg_cells]
    best_avg_idx = int(np.argmax(numeric_avgs))
    avg_cells[best_avg_idx] = f"\\textbf{{{avg_cells[best_avg_idx]}}}"
    rows_tex.append(r"\midrule")
    rows_tex.append("\\textbf{Average}$^{\\dagger}$    & " + " & ".join(avg_cells) + r" \\")

    return "\n".join(rows_tex)

print("F1-score, RF classifier:")
print(make_latex_table_metric("RF", "F1"))

F1-score, RF classifier:
Credit Default    & N/A & N/A & N/A & N/A & \textbf{0.6791} & 0.6784 & 0.6746 & 0.6758 & \underline{0.6786} \\
Fraud Detection   & N/A & N/A & N/A & N/A & \textbf{0.8992} & 0.8857 & 0.7341 & N/A & \underline{0.8934} \\
Pima Diabetes     & N/A & N/A & N/A & N/A & \textbf{0.7057} & \underline{0.7057} & 0.7038 & 0.6952 & 0.7053 \\
IBM HR Attrition  & N/A & N/A & N/A & N/A & \textbf{0.5545} & 0.5439 & 0.5189 & \underline{0.5473} & 0.5345 \\
Churn             & N/A & N/A & N/A & N/A & 0.8655 & 0.8502 & 0.8461 & \underline{0.8674} & \textbf{0.8732} \\
Ecoli             & N/A & N/A & N/A & N/A & \underline{0.8671} & 0.8671 & 0.7967 & N/A & \textbf{0.8890} \\
Satellite         & N/A & N/A & N/A & N/A & 0.7814 & \textbf{0.7924} & 0.7678 & \underline{0.7835} & 0.7651 \\
SECOM             & N/A & N/A & N/A & N/A & \textbf{0.4830} & \underline{0.4830} & 0.4830 & 0.4830 & 0.4830 \\
Thyroid Sick      & N/A & N/A & N/A & N/A & \textbf{0.9348} & 0.8839 & 0.8976 & 0.9161 & \und

In [18]:
"""
Appendix F data: PR-AUC, G-mean, Balanced Accuracy, RF classifier,
all 16 datasets, all 9 methods. Executed in the same session as the
verified `unified` re-integration above.
"""

DATASET_ORDER = [
    "credit_default", "fraud_detection", "pima_diabetes", "ibm_attrition",
    "churn", "ecoli", "satellite", "secom", "thyroid_sick", "wine_quality",
    "yeast_me2", "mammography", "abalone_19", "pageblocks",
    "protein_homo", "unsw_nb15",
]
DATASET_LABELS = {
    "credit_default": "Credit Default", "fraud_detection": "Fraud Detection",
    "pima_diabetes": "Pima Diabetes", "ibm_attrition": "IBM HR Attrition",
    "churn": "Churn", "ecoli": "Ecoli", "satellite": "Satellite",
    "secom": "SECOM", "thyroid_sick": "Thyroid Sick", "wine_quality": "Wine Quality",
    "yeast_me2": "Yeast ME2", "mammography": "Mammography", "abalone_19": "Abalone 19",
    "pageblocks": "PageBlocks", "protein_homo": "Protein Homology", "unsw_nb15": "UNSW-NB15",
}
METHOD_ORDER = ["None", "SMOTE", "ADASYN", "G-SMOTE", "KMeans_CTGAN",
                "CTGAN_MOS", "ctdGAN", "CTAB-GAN+", "Q-G-CTGAN"]

def make_latex_table_metric(classifier, metric_col):
    rows_tex = []
    col_sums = {m: [] for m in METHOD_ORDER}

    for ds in DATASET_ORDER:
        sub = unified[(unified["classifier"] == classifier) & (unified["dataset"] == ds)]
        vals = {}
        for m in METHOD_ORDER:
            row = sub[sub["method"] == m]
            if len(row) and pd.notna(row[metric_col].iloc[0]):
                vals[m] = float(row[metric_col].iloc[0])
            else:
                vals[m] = None
            if vals[m] is not None:
                col_sums[m].append(vals[m])

        present = {m: v for m, v in vals.items() if v is not None}
        ranked = sorted(present.items(), key=lambda kv: -kv[1])
        best_m = ranked[0][0] if ranked else None
        second_m = ranked[1][0] if len(ranked) > 1 else None

        cells = []
        for m in METHOD_ORDER:
            v = vals[m]
            if v is None:
                cells.append("N/A")
            else:
                s = f"{v:.4f}"
                if m == best_m:
                    s = f"\\textbf{{{s}}}"
                elif m == second_m:
                    s = f"\\underline{{{s}}}"
                cells.append(s)
        rows_tex.append(f"{DATASET_LABELS[ds]:<18}& " + " & ".join(cells) + r" \\")

    avg_cells = []
    for m in METHOD_ORDER:
        if m == "CTAB-GAN+":
            avg_cells.append("---")
        else:
            avg_cells.append(f"{np.mean(col_sums[m]):.4f}" if col_sums[m] else "N/A")
    numeric_avgs = [float(x) if x not in ("---", "N/A") else -1 for x in avg_cells]
    best_avg_idx = int(np.argmax(numeric_avgs))
    avg_cells[best_avg_idx] = f"\\textbf{{{avg_cells[best_avg_idx]}}}"
    rows_tex.append(r"\midrule")
    rows_tex.append("\\textbf{Average}$^{\\dagger}$    & " + " & ".join(avg_cells) + r" \\")

    return "\n".join(rows_tex)


for metric_name, metric_col in [("PR-AUC", "PR_AUC"), ("G-mean", "G_mean"), ("Balanced Accuracy", "Balanced_Acc")]:
    print("=" * 80)
    print(f"{metric_name}, RF classifier:")
    print("=" * 80)
    print(make_latex_table_metric("RF", metric_col))
    print()

PR-AUC, RF classifier:
Credit Default    & \textbf{0.5544} & \underline{0.5508} & 0.5469 & 0.5455 & 0.5409 & 0.5436 & 0.5328 & 0.5486 & 0.5410 \\
Fraud Detection   & \underline{0.8341} & 0.8224 & 0.7676 & \textbf{0.8441} & 0.8019 & 0.7693 & 0.4894 & N/A & 0.8054 \\
Pima Diabetes     & 0.6980 & 0.6797 & 0.6802 & 0.7054 & \textbf{0.7174} & 0.6696 & 0.6517 & 0.6723 & \underline{0.7113} \\
IBM HR Attrition  & 0.5493 & \textbf{0.5816} & \underline{0.5783} & 0.5481 & 0.3984 & 0.4008 & 0.3804 & 0.3913 & 0.3989 \\
Churn             & 0.8658 & 0.8457 & 0.8354 & 0.8646 & 0.8598 & 0.8671 & 0.8669 & \textbf{0.8791} & \underline{0.8745} \\
Ecoli             & 0.6573 & 0.6848 & 0.6855 & 0.7380 & 0.8896 & \textbf{0.9119} & 0.8008 & N/A & \underline{0.8978} \\
Satellite         & 0.7327 & \underline{0.7341} & 0.6771 & \textbf{0.7372} & 0.6774 & 0.7124 & 0.6860 & 0.6773 & 0.6729 \\
SECOM             & 0.2014 & 0.1723 & 0.1689 & 0.1484 & \textbf{0.2298} & 0.1780 & 0.1594 & 0.1947 & \underline{0.2035} \\

In [19]:
for metric_name, metric_col in [("PR-AUC", "PR_AUC"), ("G-mean", "G_mean"), ("Balanced Accuracy", "Balanced_Acc")]:
    print("=" * 80)
    print(f"{metric_name}, RF classifier:")
    print("=" * 80)
    print(make_latex_table_metric("RF", metric_col))
    print()

PR-AUC, RF classifier:
Credit Default    & \textbf{0.5544} & \underline{0.5508} & 0.5469 & 0.5455 & 0.5409 & 0.5436 & 0.5328 & 0.5486 & 0.5410 \\
Fraud Detection   & \underline{0.8341} & 0.8224 & 0.7676 & \textbf{0.8441} & 0.8019 & 0.7693 & 0.4894 & N/A & 0.8054 \\
Pima Diabetes     & 0.6980 & 0.6797 & 0.6802 & 0.7054 & \textbf{0.7174} & 0.6696 & 0.6517 & 0.6723 & \underline{0.7113} \\
IBM HR Attrition  & 0.5493 & \textbf{0.5816} & \underline{0.5783} & 0.5481 & 0.3984 & 0.4008 & 0.3804 & 0.3913 & 0.3989 \\
Churn             & 0.8658 & 0.8457 & 0.8354 & 0.8646 & 0.8598 & 0.8671 & 0.8669 & \textbf{0.8791} & \underline{0.8745} \\
Ecoli             & 0.6573 & 0.6848 & 0.6855 & 0.7380 & 0.8896 & \textbf{0.9119} & 0.8008 & N/A & \underline{0.8978} \\
Satellite         & 0.7327 & \underline{0.7341} & 0.6771 & \textbf{0.7372} & 0.6774 & 0.7124 & 0.6860 & 0.6773 & 0.6729 \\
SECOM             & 0.2014 & 0.1723 & 0.1689 & 0.1484 & \textbf{0.2298} & 0.1780 & 0.1594 & 0.1947 & \underline{0.2035} \\

In [20]:
"""
Print the full PR-AUC, G-mean, Balanced Accuracy tables (RF classifier,
16 datasets, 9 methods) directly from the saved CSV, with no reliance
on any variable left over in memory from earlier cells.
"""

import pandas as pd
import numpy as np

# Reload from disk to guarantee a clean, verified source
unified = pd.read_csv("./results/05_unified_results.csv", keep_default_na=False)

DATASET_ORDER = [
    "credit_default", "fraud_detection", "pima_diabetes", "ibm_attrition",
    "churn", "ecoli", "satellite", "secom", "thyroid_sick", "wine_quality",
    "yeast_me2", "mammography", "abalone_19", "pageblocks",
    "protein_homo", "unsw_nb15",
]
METHOD_ORDER = ["None", "SMOTE", "ADASYN", "G-SMOTE", "KMeans_CTGAN",
                "CTGAN_MOS", "ctdGAN", "CTAB-GAN+", "Q-G-CTGAN"]

def print_full_metric_table(classifier, metric_col):
    print(f"\n{'='*100}")
    print(f"{metric_col}, {classifier} classifier, RAW VALUES (no bold/underline, no rounding beyond source)")
    print(f"{'='*100}")

    pivot = unified[unified["classifier"] == classifier].copy()
    pivot = pivot[pivot["method"].isin(METHOD_ORDER)]
    pivot[metric_col] = pd.to_numeric(pivot[metric_col], errors="coerce")

    table = pivot.pivot_table(index="dataset", columns="method", values=metric_col, aggfunc="first")
    table = table.reindex(index=DATASET_ORDER, columns=METHOD_ORDER)

    pd.set_option("display.width", 200)
    pd.set_option("display.max_columns", 20)
    print(table.to_string())

    print("\nColumn averages:")
    print(table.mean(axis=0, skipna=True).to_string())

for metric_col in ["PR_AUC", "G_mean", "Balanced_Acc"]:
    print_full_metric_table("RF", metric_col)


PR_AUC, RF classifier, RAW VALUES (no bold/underline, no rounding beyond source)
method             None   SMOTE  ADASYN  G-SMOTE  KMeans_CTGAN  CTGAN_MOS  ctdGAN  CTAB-GAN+  Q-G-CTGAN
dataset                                                                                                
credit_default   0.5544  0.5508  0.5469   0.5455        0.5409     0.5436  0.5328     0.5486     0.5410
fraud_detection  0.8341  0.8224  0.7676   0.8441        0.8019     0.7693  0.4894        NaN     0.8054
pima_diabetes    0.6980  0.6797  0.6802   0.7054        0.7174     0.6696  0.6517     0.6723     0.7113
ibm_attrition    0.5493  0.5816  0.5783   0.5481        0.3984     0.4008  0.3804     0.3913     0.3989
churn            0.8658  0.8457  0.8354   0.8646        0.8598     0.8671  0.8669     0.8791     0.8745
ecoli            0.6573  0.6848  0.6855   0.7380        0.8896     0.9119  0.8008        NaN     0.8978
satellite        0.7327  0.7341  0.6771   0.7372        0.6774     0.7124  0.6860     

In [21]:
"""
Print the PR-AUC table only (RF classifier, 16 datasets, 9 methods),
directly from the saved CSV.
"""

import pandas as pd

unified = pd.read_csv("./results/05_unified_results.csv", keep_default_na=False)

DATASET_ORDER = [
    "credit_default", "fraud_detection", "pima_diabetes", "ibm_attrition",
    "churn", "ecoli", "satellite", "secom", "thyroid_sick", "wine_quality",
    "yeast_me2", "mammography", "abalone_19", "pageblocks",
    "protein_homo", "unsw_nb15",
]
METHOD_ORDER = ["None", "SMOTE", "ADASYN", "G-SMOTE", "KMeans_CTGAN",
                "CTGAN_MOS", "ctdGAN", "CTAB-GAN+", "Q-G-CTGAN"]

pivot = unified[unified["classifier"] == "RF"].copy()
pivot = pivot[pivot["method"].isin(METHOD_ORDER)]
pivot["PR_AUC"] = pd.to_numeric(pivot["PR_AUC"], errors="coerce")

table = pivot.pivot_table(index="dataset", columns="method", values="PR_AUC", aggfunc="first")
table = table.reindex(index=DATASET_ORDER, columns=METHOD_ORDER)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)

print("PR_AUC, RF classifier, RAW VALUES")
print(table.to_string())
print()
print("Column averages:")
print(table.mean(axis=0, skipna=True).to_string())

PR_AUC, RF classifier, RAW VALUES
method             None   SMOTE  ADASYN  G-SMOTE  KMeans_CTGAN  CTGAN_MOS  ctdGAN  CTAB-GAN+  Q-G-CTGAN
dataset                                                                                                
credit_default   0.5544  0.5508  0.5469   0.5455        0.5409     0.5436  0.5328     0.5486     0.5410
fraud_detection  0.8341  0.8224  0.7676   0.8441        0.8019     0.7693  0.4894        NaN     0.8054
pima_diabetes    0.6980  0.6797  0.6802   0.7054        0.7174     0.6696  0.6517     0.6723     0.7113
ibm_attrition    0.5493  0.5816  0.5783   0.5481        0.3984     0.4008  0.3804     0.3913     0.3989
churn            0.8658  0.8457  0.8354   0.8646        0.8598     0.8671  0.8669     0.8791     0.8745
ecoli            0.6573  0.6848  0.6855   0.7380        0.8896     0.9119  0.8008        NaN     0.8978
satellite        0.7327  0.7341  0.6771   0.7372        0.6774     0.7124  0.6860     0.6773     0.6729
secom            0.2014  0.172

In [22]:
for metric_col in ["G_mean", "Balanced_Acc"]:
    pivot = unified[unified["classifier"] == "RF"].copy()
    pivot = pivot[pivot["method"].isin(METHOD_ORDER)]
    pivot[metric_col] = pd.to_numeric(pivot[metric_col], errors="coerce")
    table = pivot.pivot_table(index="dataset", columns="method", values=metric_col, aggfunc="first")
    table = table.reindex(index=DATASET_ORDER, columns=METHOD_ORDER)
    print(f"\n{metric_col}, RF classifier, RAW VALUES")
    print(table.to_string())


G_mean, RF classifier, RAW VALUES
method             None   SMOTE  ADASYN  G-SMOTE  KMeans_CTGAN  CTGAN_MOS  ctdGAN  CTAB-GAN+  Q-G-CTGAN
dataset                                                                                                
credit_default   0.6972  0.6939  0.6991   0.5939        0.5899     0.5878  0.5863     0.5817     0.5882
fraud_detection  0.9025  0.9242  0.9323   0.9385        0.9040     0.8813  0.8211        NaN     0.9077
pima_diabetes    0.7497  0.7517  0.7592   0.7573        0.6942     0.6942  0.6831     0.6948     0.6998
ibm_attrition    0.5228  0.5882  0.5674   0.3211        0.3512     0.3311  0.2639     0.3320     0.3102
churn            0.8800  0.8811  0.8844   0.8358        0.7958     0.7694  0.7633     0.7988     0.8026
ecoli            0.7816  0.7918  0.8207   0.7100        0.8433     0.8433  0.7303        NaN     0.8481
satellite        0.8415  0.8524  0.8670   0.8690        0.7806     0.7332  0.6996     0.8549     0.7150
secom            0.0000  0.22